# Контест. Дискретные случайные величины

## Вариант 2

#### Импорты и настройка

В этой ячейке собраны все необходимые импорты, конфигурация графиков, RNG и константы задачи.  
**Ничего менять не нужно — просто запусти ячейку.**

In [1]:
from abc import ABC, abstractmethod

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import rv_discrete

# ── RNG для воспроизводимости ──
RNG = np.random.default_rng(42)

# ── Константы задачи ──
N_SUPPLIERS = 50  # число поставщиков в воронке
PRICE_MIN = 1  # минимальная цена
PRICE_MAX = 100  # максимальная цена
N_TRIALS = 5_000  # число симуляций для оптимизации

# ── Настройка графиков ──
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "grid.linestyle": "--",
        "grid.linewidth": 0.6,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "legend.framealpha": 0.85,
        "legend.edgecolor": "#cccccc",
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

COLORS = {
    "economy": "#2c7bb6",
    "standard": "#fdae61",
    "premium": "#d7191c",
    "mixture": "#1b9e77",
    "cdf": "#7570b3",
    "ref": "#555555",
}

print("Импорты и константы загружены.")
print(
    f"N = {N_SUPPLIERS},  цена ∈ [{PRICE_MIN}, {PRICE_MAX}],  симуляций = {N_TRIALS:,}"
)

Импорты и константы загружены.
N = 50,  цена ∈ [1, 100],  симуляций = 5,000


## Оптимизация закупок

### Контекст
Lead Data Scientist в отделе закупок крупной компании выбирает одного поставщика из воронки в $N = 50$.
После каждого раунда переговоров поставщик называет цену контракта — от 1 до 100 у.е. Поставщики приходят строго по одному. Если постащику отказали, он не возвращается с новым предложением.

Цель — **минимизировать** ожидаемую цену контракта.

### О рынке поставщиков
Рынок состоит из трёх сегментов: economy (10%), standard (40%) и premium (50%).
Распределения цен внутри сегментов характеризуются весовыми функциями:

* **Economy**: $w_E(x) = \exp\left(-\frac{|x-30|}{5}\right)$,

* **Standard**: $w_S(x) = \exp\left(-\frac{|x-65|}{6}\right)$,

* **Premium**: $w_P(x) = \exp\left(-\frac{|x-90|}{4}\right)$.

### Стратегии закупки
1) Решение классической задачи об оптимальной остановке (англоязычный вариант — *secretary problem*).
    * Пропустить первые $k$ поставщиков (запомнив минимальную цену среди них =: $m$). Затем выбрать первого поставщика, чья цена строго меньше $m$. Если такого не нашлось до конца списка — выбрать последнего.
2) Поскольку известно истинное распределение рынка, не нужно тратить первых $k$ поставщиков на «разведку». Можно заранее установить жёсткий порог приемлемости $T$.
    * Выбрать первого поставщика, чья цена $\le T$. Если такого не нашлось до конца списка — выбрать последнего.
3) Baseline
    * Выбрать первого встречного.


### **Задача**

Смоделировать распределение цен поставщиков, написать симулятор рынка через Inverse Transform Sampling и найти оптимальные параметры для двух стратегий закупки и сравнить с baseline.

## Правила контеста

* Архитектура решения, вызовы функций и код графиков *уже реализованы*. Тебе нужно реализовать только **логику расчетов** в тех местах кода, где стоит комментарий `Твой код здесь`.  
* **Нельзя** создавать новые ячейки, писать код вне явно обозначенных для этого мест, править уже написанный код проверок, запусков симуляций и визуализации.   
* Используй только библиотеку `numpy`. Не пользуйся функциями других библиотек.

## Распределения сегментов [2 балла]

Цена контракта поставщика внутри сегмента — дискретная случайная величина на $\{1, 2, \dots, 100\}$.
Распределение задаётся весовой функцией $w_E, w_S$ и $w_P$ соответственно.

- Отнормируй распределения внутри грейдов в функцию вероятности (PMF).

*Подсказка*: вычислить сумму всех цен $\Sigma_{i=1}^{100}w(x_i)$ и разделить на неё вес каждой цены.



- Базовый класс `BaseSegmentDistribution` определяет набор методов для задания распределения цен поставщиков одного сегмента. Для каждого сегмента будет создан свой класс-наследник. Реализуй в базовом классе методы `_ppf` (задает квантильную функцию) и `_rvs` (генерирует выборку).


In [2]:
class BaseSegmentDistribution(rv_discrete):
    """Базовый класс для распределения цены одного сегмента.

    Наследник должен реализовать _pmf(k).
    """

    # Требование scipy.stats.rv_discrete — обязательно переопределить __new__
    def __new__(cls, *args, **kwargs):
        return super().__new__(cls)

    def __init__(self, name: str = "segment") -> None:
        self.support = np.arange(PRICE_MIN, PRICE_MAX + 1, dtype=int)
        super().__init__(a=PRICE_MIN, b=PRICE_MAX, name=name)

    def _ppf(self, q: float | list[float] | np.ndarray[float]):
        """
        Квантильная функция.

        Параметры:
            q (float или массив float в диапазоне [0, 1]): уровень квантиля.
               Пример: 0.5 (медиана), 0.95 (95-й перцентиль),
               или np.array([0.25, 0.75]).

        Возвращает:
            int или np.ndarray[int]: Значение цены (квантиль).
                Пример: для q=0.5 вернет медиану, для q=[0.25, 0.75]
                вернет np.array([34, 44]).
        """
        support = self.support  # np.array([1, 2, ..., 100]) — носитель
        _pmf = self.pmf  # callable: pmf(k) → P(X = k)
        cdf = self.cdf  # callable: cdf(x) → F(x) = P(X ≤ x)

        # Твой код здесь
        ans = []
        if type(q) is float:
            assert 0.0 <= q <= 1.0, "q не может выходить за рамки [0, 1]"
            for i in support:
                if cdf(i) >= q:
                    return i
            return 0

        for qq in q:
            assert 0.0 <= qq <= 1.0, "qq не может выходить за рамки [0, 1]"
            for i in support:
                if cdf(i) >= qq:
                    ans.append(i)
            ans.append(0)

        if len(ans) == 1:
            assert len(ans) == len(q)
            return ans[0]
        return ans

    def _rvs(self, size: int, random_state: np.random.Generator):
        """
        Генератор случайных выборок методом обратного преобразования
        (inverse transform sampling).

        Параметры:
            size (int): размер выборки (количество генерируемых скоров).
            random_state (np.random.Generator): генератор псевдослучайных чисел для воспроизводимости.

        Возвращает:
            np.ndarray[int]: массив скоров формы (size,).
                Пример: size=5 -> np.array([42, 38, 44, 35, 40]).
        """
        ppf = self.ppf  # callable: ppf(u) → квантиль уровня u ∈ [0,1]

        # Твой код здесь
        return [ppf(i) for i in random_state.uniform(size=size)]

- Реализуй метод `_pmf(k)`, который задает функцию вероятностей для распределения цен, в каждом классе-наследнике.

Весовые функции:
- **Economy**:  $w(x) = \exp\bigl(-|x-30| \,/\, 5\bigr)$
- **Standard**:  $w(x) = \exp\bigl(-|x-65| \,/\, 6\bigr)$
- **Premium**:  $w(x) = \exp\bigl(-|x-90| \,/\, 4\bigr)$


In [3]:
class EconomyDistribution(BaseSegmentDistribution):
    """Распределение цены Economy-поставщиков."""

    def __init__(self) -> None:
        super().__init__(name="economy")

    def _pmf(self, k: int | np.ndarray[int]):
        """
        Функция вероятностей (PMF) распределения Economy.

        Принимает:
            k (int или np.ndarray[int]): значение скора.

        Возвращает:
            float или np.ndarray[float]: вероятность для каждого значения k.
                Пример: k = 30 -> float, P(X=30) ≈ 0.045 (пик)
                Пример: k = [20, 30, 40] -> np.array([≈0.016, ≈0.045, ≈0.016])
                Пример: k = 0 или k = 101 -> 0.0 (вне носителя)
        """
        _support = self.support  # np.array([1, 2, ..., 100]) — носитель

        # Твой код здесь
        if type(k) is int:
            return np.exp(-np.abs(k - 30) / 5) / 100
        ans = []
        for kk in k:
            ans.append(np.exp(-np.abs(kk - 30) / 5))
        sm = sum(ans)
        for i, _ in enumerate(ans):
            ans[i] /= sm
        assert abs(sum(ans) - 1.0) < 1e-12, "Сумма PMF != 1"
        return ans


class StandardDistribution(BaseSegmentDistribution):
    """Распределение цены Standard-поставщиков."""

    def __init__(self) -> None:
        super().__init__(name="standard")

    def _pmf(self, k):
        """
        Функция вероятностей (PMF) распределения Standard.

        Принимает:
            k (int или np.ndarray[int]): значение скора.

        Возвращает:
            float или np.ndarray[float]: вероятность для каждого значения k.
                Пример: k = 65 -> P(X=65) ≈ 0.034 (пик)
                Пример: k = [55, 65, 75] -> np.array([≈0.018, ≈0.034, ≈0.018])
        """

        # Твой код здесь
        if type(k) is int:
            return np.exp(-np.abs(k - 65) / 6) / 100
        ans = []
        for kk in k:
            ans.append(np.exp(-np.abs(kk - 65) / 6))
        sm = sum(ans)
        for i, _ in enumerate(ans):
            ans[i] /= sm
        assert abs(sum(ans) - 1.0) < 1e-12, "Сумма PMF != 1"
        return ans


class PremiumDistribution(BaseSegmentDistribution):
    """Распределение цены Premium-поставщиков."""

    def __init__(self) -> None:
        super().__init__(name="premium")

    def _pmf(self, k):
        """
        Функция вероятности (PMF) распределения Premium.

        Принимает:
            k (int или np.ndarray[int]): значение скора.

        Возвращает:
            float или np.ndarray[float]: вероятность для каждого значения k.
                Пример: k = 90 -> P(X=90) ≈ 0.057 (пик)
                Пример: k = [85, 90, 95] -> np.array([≈0.034, ≈0.057, ≈0.034])
        """

        # Твой код здесь
        if type(k) is int:
            return np.exp(-np.abs(k - 90) / 4) / 100
        ans = []
        for kk in k:
            ans.append(np.exp(-np.abs(kk - 90) / 4))
        sm = sum(ans)
        for i, _ in enumerate(ans):
            ans[i] /= sm
        assert abs(sum(ans) - 1.0) < 1e-12, "Сумма PMF != 1"
        return ans

### Проверка распределений сегментов

Проверим корректность PMF, квантильной функции `_ppf` и генератора `_rvs`.

*Ничего менять не нужно — просто запусти ячейку.*

In [5]:
# ── Проверка классов сегментов ──
economy = EconomyDistribution()
standard = StandardDistribution()
premium = PremiumDistribution()


# 2. Сумма PMF = 1 и PMF неотрицательна
for dist in [economy, standard, premium]:
    pmf_vals = dist.pmf(dist.support)
    assert abs(pmf_vals.sum() - 1.0) < 1e-12, (
        f"{dist.name}: сумма PMF != 1, PMF == {pmf_vals.sum()}"
    )
    assert np.all(pmf_vals >= -1e-15), f"{dist.name}: отрицательные вероятности"

# 3. CDF монотонна и CDF(100) = 1
for dist in [economy, standard, premium]:
    cdf_vals = dist.cdf(dist.support)
    assert np.all(np.diff(cdf_vals) >= -1e-15), f"{dist.name}: CDF не монотонна"
    assert abs(cdf_vals[-1] - 1.0) < 1e-12, f"{dist.name}: CDF(100) != 1"

# 4. PMF для значений вне носителя = 0
for dist in [economy, standard, premium]:
    assert np.all(dist.pmf(np.array([0, -5, 101, 200])) == 0), (
        f"{dist.name}: PMF вне носителя не равен 0"
    )

# 5. Мат. ожидания: Economy < Standard < Premium
assert economy.mean() < standard.mean() < premium.mean(), (
    f"E: {economy.mean():.1f} < {standard.mean():.1f} < {premium.mean():.1f} нарушено"
)

# 6. Медианы тоже должны быть упорядочены
assert economy.median() < standard.median() < premium.median(), (
    f"Медианы не упорядочены {economy.median()} {standard.median()} {premium.median()}"
)

# 7. _ppf: F(F⁻¹(q)) ≥ q для контрольных квантилей
for dist in [economy, standard, premium]:
    for q_test in [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]:
        x_q = int(dist._ppf(q_test))
        cdf_at_x = dist.cdf(x_q)
        assert cdf_at_x >= q_test - 1e-12, (
            f"{dist.name}: ppf({q_test})={x_q}, но CDF({x_q})={cdf_at_x:.4f} < {q_test}"
        )

# 8. _rvs: выборочное среднее близко к теоретическому
for dist in [economy, standard, premium]:
    s = dist._rvs(size=10_000, random_state=RNG)
    assert len(s) == 10_000, f"{dist.name}: неверный размер выборки"
    assert s.min() >= PRICE_MIN and s.max() <= PRICE_MAX, (
        f"{dist.name}: выборка вне носителя"
    )
    assert abs(s.mean() - dist.mean()) < 1.5, (
        f"{dist.name}: выб. среднее {s.mean():.2f} далеко от теор. {dist.mean():.2f}"
    )

print("Все проверки классов сегментов пройдены")

# ── Визуализация PMF всех трёх сегментов ──
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)
x_plot = np.arange(PRICE_MIN, PRICE_MAX + 1)

for ax, dist, color, title in [
    (axes[0], economy, COLORS["economy"], "Economy"),
    (axes[1], standard, COLORS["standard"], "Standard"),
    (axes[2], premium, COLORS["premium"], "Premium"),
]:
    ax.bar(
        x_plot,
        dist.pmf(x_plot),
        width=0.8,
        color=color,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.axvline(
        dist.mean(),
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"E = {dist.mean():.1f}",
    )
    ax.axvline(
        dist.median(),
        color="#555555",
        linestyle=":",
        linewidth=1.5,
        label=f"Med = {dist.median():.0f}",
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Цена")
    ax.set_ylabel("$\\mathbb{P}(X = k)$")
    ax.set_xlim(0, 101)
    ax.legend(loc="upper right", fontsize=8)
    ax.set_axisbelow(True)

fig.suptitle("PMF распределений сегментов", fontsize=15, fontweight="bold", y=1.03)
plt.show()

AssertionError: Медианы не упорядочены 1.0 1.0 1.0

## Класс смеси распределений — `MixtureDistribution` [1 балл]

Рынок поставщиков — *смесь* трёх сегментов. Чтобы получить рынок из условия (10% / 40% / 50%) необходимо с соответствующими весами смешать распределения Economy, Standard и Premium:

$$X_{\text{market}} \sim w_E \cdot \text{Economy} + w_S \cdot \text{Standard} + w_P \cdot \text{Premium}.$$

- Реализуй методы `_pmf`, `_ppf`, `_rvs` класса `MixtureDistribution` (наследник `rv_discrete`). Код для функций
`_ppf` и `_rvs` можно взять из `BaseSegmentDistribution`.


In [ ]:
class MixtureDistribution(rv_discrete):
    """Дискретная случайная величина — смесь распределений."""

    # Требование scipy.stats.rv_discrete
    def __new__(cls, *args, **kwargs):
        return super().__new__(cls)

    def __init__(self, distributions: list[float], weights, name="mixture") -> None:
        self.distributions = list(distributions)
        self.mixture_weights = list(weights)
        self.support = np.arange(PRICE_MIN, PRICE_MAX + 1, dtype=int)
        super().__init__(a=PRICE_MIN, b=PRICE_MAX, name=name)

    def _pmf(self, k):
        """
        Функция вероятностей (PMF) смеси распределений.

        Принимает:
            k (int или np.ndarray[int]): значение цены.

        Возвращает:
            float или np.ndarray[float]: вероятность для каждого значения k.
                Пример: k = 50 -> float
                Пример: k = [30, 70] -> np.array([P(30) ≈ 0.023, P(70) ≈ 0.010])
        """

        # Твой код здесь

    def _ppf(self, q):
        """
        Квантильная функция.

        Принимает:
            q (float или np.ndarray[float] в диапазоне [0, 1]): квантиль уровня.

        Возвращает:
            int или np.ndarray[int]: минимальное x, такое что CDF(x) >= q.
                Пример: q = 0.5 -> вернуть медиану (int)
                Пример: q = 0.95 -> вернуть 95-й перцентиль
                Пример: q = [0.25, 0.75] -> вернуть np.array([34, 44])
        """

        # Твой код здесь

    def _rvs(self, size, random_state):
        """
        Генератор случайных выборок методом обратного преобразования
        (inverse transform sampling).

        Принимает:
            size (int): размер выборки (количество генерируемых цен).
            random_state (np.random.Generator): генератор псевдослучайных чисел.

        Возвращает:
            np.ndarray[int]: массив скоров формы (size,).
                Пример: size = 5 -> np.array([42, 38, 44, 35, 40])
                Пример: size = 100 -> ndarray[int] из 100 цен
        """

        # Твой код здесь

### Проверка `MixtureDistribution` и создание рынка

*Ничего менять не нужно — просто запусти ячейку.*

In [ ]:
w_economy = 0.1
w_standard = 0.4
w_premium = 0.5

market = MixtureDistribution(
    distributions=[economy, standard, premium],
    weights=[w_economy, w_standard, w_premium],
    name="market",
)

# 1. Сумма PMF = 1
assert abs(market.pmf(market.support).sum() - 1.0) < 1e-12, "Сумма PMF смеси != 1"

# 2. CDF монотонна и заканчивается 1
cdf_vals = market.cdf(market.support)
assert np.all(np.diff(cdf_vals) >= -1e-15), "CDF смеси не монотонна"
assert abs(cdf_vals[-1] - 1.0) < 1e-12, "CDF смеси(100) != 1"

# 4. Квантили: F(F⁻¹(q)) ≥ q
for q_test in [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]:
    x_q = int(market._ppf(q_test))
    cdf_at_x = market.cdf(x_q)
    assert cdf_at_x >= q_test - 1e-12, f"ppf({q_test})={x_q}, CDF={cdf_at_x} < {q_test}"

# 5. Генерация выборки
market_sample = market.rvs(size=100_000, random_state=RNG)
assert len(market_sample) == 100_000
assert market_sample.min() >= PRICE_MIN and market_sample.max() <= PRICE_MAX
assert abs(market_sample.mean() - market.mean()) < 0.5, (
    f"Выб. среднее {market_sample.mean():.2f} далеко от теор. {market.mean():.2f}"
)

# 6. В выборке должны быть представлены все три диапазона
assert np.any(market_sample < 45), "Нет низких цен (Economy)"
assert np.any(market_sample > 80), "Нет высоких цен (Premium)"

print("Все проверки MixtureDistribution пройдены")

# ── Визуализация ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
x_plot = np.arange(PRICE_MIN, PRICE_MAX + 1)


# PMF рынка
ax0 = axes[0]
ax0.bar(
    x_plot,
    market.pmf(x_plot),
    width=0.8,
    color=COLORS["mixture"],
    alpha=0.7,
    edgecolor="white",
    linewidth=0.3,
    label="Рынок (смесь)",
)
ax0.plot(
    x_plot,
    w_economy * economy.pmf(x_plot),
    color=COLORS["economy"],
    linewidth=1.2,
    label=f"Economy × {w_economy:.2f}",
)
ax0.plot(
    x_plot,
    w_standard * standard.pmf(x_plot),
    color=COLORS["standard"],
    linewidth=1.2,
    label=f"Standard × {w_standard:.2f}",
)
ax0.plot(
    x_plot,
    w_premium * premium.pmf(x_plot),
    color=COLORS["premium"],
    linewidth=1.2,
    label=f"Premium × {w_premium:.2f}",
)
ax0.set_title("PMF рынка поставщиков", fontweight="bold")
ax0.set_xlabel("Цена")
ax0.set_ylabel("$\\mathbb{P}(X = k)$")
ax0.legend(loc="upper right", fontsize=8)
ax0.set_axisbelow(True)
ax0.set_xlim(0, 101)


# CDF рынка — ступенчатая функция
ax1 = axes[1]
x_left = float(x_plot[0]) - 1.0
ax1.hlines(0.0, x_left, float(x_plot[0]), colors=COLORS["cdf"], linewidth=2.2, zorder=3)

for i, (x_i, f_i) in enumerate(zip(x_plot, cdf_vals)):
    x_i = float(x_i)
    f_i = float(f_i)
    f_left = 0.0 if i == 0 else float(cdf_vals[i - 1])
    if f_i > f_left:
        ax1.plot(
            [x_i, x_i],
            [f_left, f_i],
            color=COLORS["cdf"],
            linestyle=(0, (2, 2)),
            linewidth=1.0,
            alpha=0.7,
            zorder=3,
        )
    x_next = float(x_plot[i + 1]) if i + 1 < len(x_plot) else x_i + 1.0
    ax1.hlines(f_i, x_i, x_next, colors=COLORS["cdf"], linewidth=1.0, zorder=3)

ax1.scatter(
    x_plot,
    cdf_vals,
    color=COLORS["cdf"],
    s=3,
    zorder=6,
    edgecolors="white",
    linewidth=0.6,
)
left_limits = np.concatenate([[0.0], cdf_vals[:-1]])
ax1.scatter(
    x_plot,
    left_limits,
    facecolors="white",
    edgecolors=COLORS["cdf"],
    s=2,
    zorder=5,
    linewidth=0.5,
)

for y_line, ls in [(0, "-."), (1, "-.")]:
    ax1.axhline(y=y_line, color="#aaaaaa", linestyle=ls, linewidth=0.7, alpha=0.5)

ax1.set_title("CDF рынка поставщиков", fontweight="bold", pad=12)
ax1.set_xlabel("$x$")
ax1.set_ylabel("$F(x) = \\mathbb{P}(X \\leq x)$")
ax1.set_axisbelow(True)
ax1.set_ylim(-0.05, 1.08)

ax1.axhline(y=0.5, color="#aaaaaa", linestyle=":", linewidth=1.0)

med = market.median()
ax1.annotate(
    f"Медиана = {med:.0f}",
    xy=(med, 0.5),
    xytext=(med + 12, 0.42),
    arrowprops={"arrowstyle": "->", "color": "#555555", "lw": 1.2},
    fontsize=10,
    color="#333333",
    bbox={
        "boxstyle": "round,pad=0.3",
        "facecolor": "white",
        "edgecolor": "#aaaaaa",
        "alpha": 0.8,
    },
)

fig.suptitle(
    "Рынок поставщиков: смесь Economy (10%) + Standard (40%) + Premium (50%)",
    fontsize=14,
    fontweight="bold",
    y=1.03,
)
plt.show()

print("  Характеристики рынка:")
print(f"    E[Цена] = {market.mean():.2f}")
print(f"    Медиана   = {market.median():.0f}")
print(f"    Std       = {market.std():.2f}")

NameError: name 'MixtureDistribution' is not defined

## Стратегии закупки [3 балла]

Реализуй три стратегии:
- `BaselineStrategy`,
- `ClassicSecretaryStrategy`,
- `ThresholdStrategy`.

Базовый класс `ProcurementStrategy` с методом `simulate` уже написан.

### Архитектура

```
ProcurementStrategy (ABC)
├── BaselineStrategy           — выбрать первого встречного
├── ClassicSecretaryStrategy   — классическая задача об оптимальной остановке (параметр k)
└── ThresholdStrategy          — пороговая стратегия (параметр T)
```

В каждом классе нужно реализовать только метод `select(scores)`, возвращающий индекс
выбранного поставщика.

In [ ]:
class ProcurementStrategy(ABC):
    """Базовый класс стратегии закупки."""

    def __init__(self, name: str) -> None:
        self.name = name

    @abstractmethod
    def select(self, scores: np.ndarray) -> int:
        """
        Выбирает индекс поставщика для закупки.

        Принимает:
            scores (np.ndarray[int]): массив цен поставщиков.

        Возвращает:
            int: индекс выбранного поставщика.
        """
        # Твой код здесь

    def simulate(
        self,
        dist: rv_discrete,
        n_suppliers: int,
        n_trials: int,
        random_state: np.random.Generator = RNG,
    ) -> np.ndarray:
        """
        Запускает симуляцию стратегии.

        Возвращает:
        np.ndarray, shape (n_trials,): цены выбранных поставщиков для каждой симуляции.
        """
        all_scores = dist.rvs(size=(n_trials, n_suppliers), random_state=random_state)
        hired = np.empty(n_trials, dtype=int)
        for i in range(n_trials):
            idx = self.select(all_scores[i])
            hired[i] = all_scores[i, idx]
        return hired


class BaselineStrategy(ProcurementStrategy):
    """Стратегия: выбрать первого встречного поставщика."""

    def __init__(self) -> None:
        super().__init__(name="Baseline")

    def select(self, scores: np.ndarray) -> int:
        """
        Выбирает индекс поставщика для закупки (baseline: первый кандидат).

        Принимает:
            scores (np.ndarray[int]): массив цен поставщиков (по порядку).
                Пример: [50, 90, 10, 70, 30]

        Возвращает:
            int: индекс выбранного поставщика (всегда 0).
        """
        # Твой код здесь


class ClassicSecretaryStrategy(ProcurementStrategy):
    """Классическая задача об оптимальной остановке (аналог задачи о разборчивой невесте).

    Алгоритм:
      1. Пропустить первых k поставщиков, запомнив минимум m.
      2. Выбрать первого после k, чья цена строго меньше m.
      3. Если такого нет до конца — выбрать последнего.
    """

    def __init__(self, k: int) -> None:
        super().__init__(name=f"ClassicSecretary(k={k})")
        self.k = k

    def select(self, scores: np.ndarray) -> int:
        """
        Выбирает индекс поставщика для закупки (классическая задача о невесте).

        Принимает:
            scores (np.ndarray[int]): массив цен поставщиков (длины N).
                Пример: [50, 90, 10, 70, 30]

        Возвращает:
            int: индекс выбранного поставщика (0 ... N-1).
                Пример: k=2, scores=[50, 90, 10, 70, 30] -> M=max(50,90)=90 -> ни один из [10,70,30] не > 90 -> вернуть 4 (последний)
                Пример: k=1, scores=[50, 90, 10, 70, 30] -> M=50 -> первый > 50 это 90 на индексе 1 -> вернуть 1
                Пример: k=0, scores=[50, 90, 10, 70, 30] -> M=-∞ -> первое же значение > -∞ это 50 -> вернуть 0
        """
        len(scores)  # int — общее число поставщиков

        # Твой код здесь


class ThresholdStrategy(ProcurementStrategy):
    """Пороговая стратегия: выбрать первого поставщика с ценой ≤ T.

    Алгоритм:
      1. Выбрать первого, чья цена ≤ T.
      2. Если такого нет до конца — выбрать последнего.
    """

    def __init__(self, threshold: int) -> None:
        super().__init__(name=f"Threshold(T={threshold})")
        self.threshold = threshold

    def select(self, scores: np.ndarray) -> int:
        """
        Выбирает индекс поставщика для закупки (пороговая стратегия: первый со скором >= T).

        Принимает:
            scores (np.ndarray[int]): массив цен поставщиков (длины N).
                Пример: [45, 95, 80, 60, 88]

        Возвращает:
            int: индекс первого кандидата со скором >= T, или n-1 (последний), если таких нет.
                Пример: T=80, scores=[45, 95, 80, 60, 88] -> 1 (95 >= 80)
                Пример: T=99, scores=[45, 95, 80, 60, 88] -> все < 99 -> вернуть 4 (последний)
                Пример: T=1,  scores=[45, 95, 80, 60, 88] -> все >= 1 -> 0
        """
        len(scores)  # int — общее число поставщиков

        # Твой код здесь

### Проверка стратегий на детерминированных примерах

In [ ]:
# ── Детерминированные тесты ──
test_scores = np.array([50, 90, 10, 70, 30])

# 1. Baseline: всегда выбирает первого
baseline = BaselineStrategy()
assert baseline.select(test_scores) == 0, "Baseline должен выбрать индекс 0"
print("BaselineStrategy: тест пройден")

# 2. ClassicSecretary: k=2, первые два = [50, 90], m=50
#    Ищем первый < 50: это 10 на индексе 2
sec_k2 = ClassicSecretaryStrategy(k=2)
assert sec_k2.select(test_scores) == 2, (
    f"ClassicSecretary(k=2): ожидался индекс 2, получен {sec_k2.select(test_scores)}"
)

# 3. ClassicSecretary: k=0 (m = +∞, берём первого же)
sec_k0 = ClassicSecretaryStrategy(k=0)
assert sec_k0.select(test_scores) == 0, "ClassicSecretary(k=0): ожидался индекс 0"

# 4. ClassicSecretary: k=1, первые = [50], m=50; первый < 50 — 10 на индексе 2
sec_k1 = ClassicSecretaryStrategy(k=1)
assert sec_k1.select(test_scores) == 2, "ClassicSecretary(k=1): ожидался индекс 2"
print("ClassicSecretaryStrategy: все тесты пройдены")

# 5. Threshold: T=40, первый ≤ 40 — 10 на индексе 2
thr = ThresholdStrategy(threshold=40)
assert thr.select(test_scores) == 2, "Threshold(T=40): ожидался индекс 2"

# 6. Threshold: T=5, все цены > 5 → последний (индекс 4)
thr_low = ThresholdStrategy(threshold=5)
assert thr_low.select(test_scores) == 4, "Threshold(T=5): ожидался индекс 4"

# 7. Threshold: T=100, все цены ≤ 100 → индекс 0
thr_hi = ThresholdStrategy(threshold=100)
assert thr_hi.select(test_scores) == 0, "Threshold(T=100): ожидался индекс 0"
print("ThresholdStrategy: все тесты пройдены")

# ── Проверка simulate ──
n_test_trials = 100
hired_bl = baseline.simulate(market, N_SUPPLIERS, n_test_trials, RNG)
assert len(hired_bl) == n_test_trials, "Неверная длина результата simulate"
assert hired_bl.min() >= PRICE_MIN and hired_bl.max() <= PRICE_MAX
print(
    f"simulate: {n_test_trials} симуляций Baseline прошли; "
    f"средняя цена = {hired_bl.mean():.2f}"
)

## Оптимизация параметров стратегий — `StrategyOptimizer` [3 балла]

Реализуй класс `StrategyOptimizer`.

Напиши два метода:
- `evaluate(**params)` — создает стратегию, запускает simulate, возвращает среднюю цену.
- `grid_search(param_grid)` — перебирает все комбинации параметров по сетке,
  возвращает лучшие параметры и полную историю результатов.

In [ ]:
class StrategyOptimizer:
    """Поиск оптимальных параметров стратегии закупки."""

    def __init__(
        self,
        strategy_class,
        dist: rv_discrete,
        n_suppliers: int,
        n_trials: int,
        random_state: np.random.Generator = RNG,
    ) -> None:
        self.strategy_class = strategy_class
        self.dist = dist
        self.n_suppliers = n_suppliers
        self.n_trials = n_trials
        self.random_state = random_state

    def evaluate(self, **params) -> float:
        """
        Оценить стратегию с заданными параметрами.

        Принимает:
            **params: параметры для инициализации стратегии.

        Возвращает:
            float: средняя цена выбранных поставщиков по результатам n_trials симуляций.
        """
        strategy_class = self.strategy_class  # класс стратегии
        strategy_class(**params)  # объект стратегии
        #   np.ndarray, shape (n_trials,)

        # Твой код здесь

    def grid_search(self, param_grid: dict) -> dict:
        """
        Поиск оптимальных параметров методом полного перебора (Grid Search).

        Принимает:
            param_grid (dict): словарь вида {'имя_параметра': iterable_значений}.
                Пример: param_grid = {'k': range(0, 50)} -> перебор k=0,1,...,49
                Пример: param_grid = {'threshold': range(1, 101)} -> перебор T=1,2,...,100

        Возвращает:
            dict: словарь с результатами поиска, содержащий ключи:
                'best_params' (dict): лучшие параметры, например {'k': 12}.
                'best_score' (float): лучшая средняя цена, например 54.32.
                'all_results' (list[dict]): все результаты вычислений, например
                    [{'params': {'k': 0}, 'score': 44.0}, {'params': {'k': 1}, 'score': 44.5}, ...].
        """

        # Твой код здесь

### Проверка `StrategyOptimizer`

*Ничего менять не нужно — просто запусти ячейку.*

In [ ]:
# ── Тест на Baseline (без параметров) ──
opt_baseline = StrategyOptimizer(
    BaselineStrategy, market, N_SUPPLIERS, n_trials=2_000, random_state=RNG
)
score_bl = opt_baseline.evaluate()
assert 60 < score_bl < 90, (
    f"Средняя цена Baseline {score_bl:.2f} вне ожидаемого диапазона"
)
print(f"Baseline средняя цена (2 000 симуляций): {score_bl:.2f}")

# ── Grid search на пустой сетке ──
result_bl = opt_baseline.grid_search({})
assert len(result_bl["all_results"]) == 1, "Должен быть 1 результат для пустой сетки"
print(f"Grid search Baseline: best_score = {result_bl['best_score']:.2f}")
print("StrategyOptimizer: базовая проверка пройдена")

### Запуск оптимизации

Найдем оптимальные параметры для Classic Secretary (по $k$) и Threshold (по $T$).

*Ничего менять не нужно — просто запусти ячейку.*

In [ ]:
N_OPT_TRIALS = N_TRIALS  # 5000 симуляций на точку

# ── Оптимизация ClassicSecretaryStrategy: перебор k от 0 до N-1 ──
opt_secretary = StrategyOptimizer(
    ClassicSecretaryStrategy, market, N_SUPPLIERS, N_OPT_TRIALS, random_state=RNG
)

print("Оптимизация ClassicSecretaryStrategy...")
result_sec = opt_secretary.grid_search({"k": range(N_SUPPLIERS)})
print(
    f"  Лучший k = {result_sec['best_params']['k']},  "
    f"средняя цена = {result_sec['best_score']:.2f}"
)
print()

# ── Оптимизация ThresholdStrategy: перебор T от 1 до 100 ──
opt_threshold = StrategyOptimizer(
    ThresholdStrategy, market, N_SUPPLIERS, N_OPT_TRIALS, random_state=RNG
)

print("Оптимизация ThresholdStrategy...")
result_thr = opt_threshold.grid_search({"threshold": range(PRICE_MIN, PRICE_MAX + 1)})
print(
    f"  Лучший T = {result_thr['best_params']['threshold']},  "
    f"средняя цена = {result_thr['best_score']:.2f}"
)

## Финальный анализ и сравнение стратегий
  
Сравним все три стратегии с оптимальными параметрами и построим графики.

*Ничего менять не нужно — просто запусти ячейку.*


In [ ]:
# ── Извлекаем результаты оптимизации ──
best_k = result_sec["best_params"]["k"]
best_T = result_thr["best_params"]["threshold"]

# Создаём стратегии с оптимальными параметрами
baseline_opt = BaselineStrategy()
secretary_opt = ClassicSecretaryStrategy(k=best_k)
threshold_opt = ThresholdStrategy(threshold=best_T)

# ── Большая симуляция для финального сравнения ──
N_FINAL = 100_000
print(f"Финальная симуляция: {N_FINAL:,} испытаний для каждой стратегии...")

hired_baseline = baseline_opt.simulate(market, N_SUPPLIERS, N_FINAL, RNG)
hired_secretary = secretary_opt.simulate(market, N_SUPPLIERS, N_FINAL, RNG)
hired_threshold = threshold_opt.simulate(market, N_SUPPLIERS, N_FINAL, RNG)

# ── Сводная таблица ──
print()
print("=" * 70)
print("  СРАВНЕНИЕ СТРАТЕГИЙ ЗАКУПКИ")
print("=" * 70)
print(f"  {'Стратегия':<30s}  {'Параметр':>10s}  {'Средняя цена':>14s}  {'Std':>8s}")
print(f"  {'─' * 30}  {'─' * 10}  {'─' * 14}  {'─' * 8}")
print(
    f"  {'Baseline':<30s}  {'—':>10s}  {hired_baseline.mean():>14.2f}  "
    f"{hired_baseline.std():>8.2f}"
)
print(
    f"  {'Classic Secretary':<30s}  {f'k = {best_k}':>10s}  "
    f"{hired_secretary.mean():>14.2f}  {hired_secretary.std():>8.2f}"
)
print(
    f"  {'Threshold':<30s}  {f'T = {best_T}':>10s}  "
    f"{hired_threshold.mean():>14.2f}  {hired_threshold.std():>8.2f}"
)
print(
    f"  {'Теоретическое E рынка':<30s}  {'—':>10s}  {market.mean():>14.2f}  "
    f"{market.std():>8.2f}"
)
print()

saving_sec = hired_baseline.mean() - hired_secretary.mean()
saving_thr = hired_baseline.mean() - hired_threshold.mean()
print("  Экономия относительно Baseline:")
print(
    f"    Classic Secretary:  {saving_sec:.2f}  "
    f"({saving_sec / hired_baseline.mean() * 100:.1f}%)"
)
print(
    f"    Threshold:          {saving_thr:.2f}  "
    f"({saving_thr / hired_baseline.mean() * 100:.1f}%)"
)

# ── Графики кривых оптимизации ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5.0), constrained_layout=True)

# График 1: Средняя цена vs k
ax0 = axes[0]
k_values = [r["params"]["k"] for r in result_sec["all_results"]]
scores_sec = [r["score"] for r in result_sec["all_results"]]
ax0.plot(
    k_values,
    scores_sec,
    color=COLORS["mixture"],
    linewidth=2.0,
    marker=".",
    markersize=3,
)
ax0.axhline(
    y=hired_baseline.mean(),
    color=COLORS["ref"],
    linestyle="--",
    linewidth=1.5,
    label=f"Baseline ({hired_baseline.mean():.1f})",
)
ax0.axvline(
    x=best_k,
    color=COLORS["premium"],
    linestyle=":",
    linewidth=1.5,
    label=f"Оптимум k = {best_k}",
)
ax0.scatter(
    [best_k],
    [result_sec["best_score"]],
    color=COLORS["premium"],
    s=100,
    zorder=10,
    edgecolors="white",
    linewidth=1.5,
)
ax0.set_title("Classic Secretary: средняя цена vs k", fontweight="bold")
ax0.set_xlabel("Параметр $k$ (сколько пропустить)")
ax0.set_ylabel("Средняя цена выбранного")
ax0.legend(loc="lower right")
ax0.set_axisbelow(True)

# График 2: Средняя цена vs T
ax1 = axes[1]
t_values = [r["params"]["threshold"] for r in result_thr["all_results"]]
scores_thr = [r["score"] for r in result_thr["all_results"]]
ax1.plot(
    t_values,
    scores_thr,
    color=COLORS["standard"],
    linewidth=2.0,
    marker=".",
    markersize=3,
)
ax1.axhline(
    y=hired_baseline.mean(),
    color=COLORS["ref"],
    linestyle="--",
    linewidth=1.5,
    label=f"Baseline ({hired_baseline.mean():.1f})",
)
ax1.axvline(
    x=best_T,
    color=COLORS["premium"],
    linestyle=":",
    linewidth=1.5,
    label=f"Оптимум T = {best_T}",
)
ax1.scatter(
    [best_T],
    [result_thr["best_score"]],
    color=COLORS["premium"],
    s=100,
    zorder=10,
    edgecolors="white",
    linewidth=1.5,
)
ax1.set_title("Threshold: средняя цена vs порог T", fontweight="bold")
ax1.set_xlabel("Порог $T$")
ax1.set_ylabel("Средняя цена выбранного")
ax1.legend(loc="lower right")
ax1.set_axisbelow(True)

fig.suptitle(
    f"Оптимизация стратегий закупки (N = {N_SUPPLIERS}, "
    f"{N_OPT_TRIALS:,} симуляций на точку)",
    fontsize=14,
    fontweight="bold",
    y=1.03,
)
plt.show()

# ── Гистограммы цен выбранных ──
fig2, axes2 = plt.subplots(
    1, 3, figsize=(18, 4.5), constrained_layout=True, sharey=True
)
bins = np.arange(PRICE_MIN - 0.5, PRICE_MAX + 1.5, 1)

for ax, data, name, color in [
    (axes2[0], hired_baseline, "Baseline", "#888888"),
    (axes2[1], hired_secretary, f"Classic Secretary (k={best_k})", COLORS["mixture"]),
    (axes2[2], hired_threshold, f"Threshold (T={best_T})", COLORS["standard"]),
]:
    ax.hist(
        data,
        bins=bins,
        density=True,
        color=color,
        alpha=0.75,
        edgecolor="white",
        linewidth=0.3,
    )
    ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1.8)
    ax.set_title(f"{name}\nСреднее = {data.mean():.2f}", fontweight="bold", fontsize=10)
    ax.set_xlabel("Цена")
    ax.set_xlim(0, 101)
    ax.set_axisbelow(True)

axes2[0].set_ylabel("Плотность")
fig2.suptitle(
    "Распределение цены выбранного поставщика", fontsize=14, fontweight="bold", y=1.03
)
plt.show()

## Выводы [1 балл]

**1.** ...

**2.** ...

**3.** ...

**4.** ...

**...**